# LLM-Bias-Miner: Bias Mining Walkthrough

This notebook walks through all four bias mining methods step by step.
Uses sample data — **no GPU or API key required**.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.collectors.sample_generator import generate_sample_data
from src.miners.association_miner import AssociationBiasMiner
from src.miners.subgroup_miner import SubgroupBiasMiner
from src.miners.statistical_tester import StatisticalBiasTester
from src.metrics.fairness_metrics import FairnessMetrics
from src.visualization.bias_plots import BiasPlotter

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Generate Sample Data

The sample generator creates synthetic LLM responses with **embedded bias patterns** calibrated to match real-world observations from Gender Shades, BBQ, and DecodingTrust.

In [ ]:
df = generate_sample_data(n_prompts=1000, seed=42)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Quick look at demographic distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['gender', 'race_ethnicity', 'age_group']):
    df[col].value_counts().plot(kind='bar', ax=ax, title=col)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 2. Association Rule Mining

**Goal:** Find co-occurrence patterns like `{race=Black, category=criminal_justice} → {toxicity=high}`.

We treat each (prompt, response) pair as a transaction, with demographic attributes and response features as items.

In [ ]:
assoc_miner = AssociationBiasMiner(
    min_support=0.02,
    min_confidence=0.4,
    min_lift=1.2,
    require_demographic_antecedent=True
)

rules = assoc_miner.mine(df)
rules_df = assoc_miner.rules_to_dataframe(rules)
print(assoc_miner.summarize(rules))

In [ ]:
# View top rules
if len(rules_df) > 0:
    display(rules_df.head(15))
else:
    print("No rules found at current thresholds. Try lowering min_support.")

## 3. Subgroup Discovery

**Goal:** Find intersectional demographic subgroups where the LLM's behavior deviates most from the overall population.

Uses beam search to explore the combinatorial space of attribute intersections efficiently.

In [ ]:
sg_miner = SubgroupBiasMiner(
    beam_width=10,
    max_depth=3,
    min_subgroup_size=20,
    quality_measure='wracc'
)

# Search for biased subgroups on multiple metrics
for target in ['toxicity_score', 'sentiment_negative', 'regard_negative']:
    print(f"\n{'='*50}")
    subgroups = sg_miner.discover(df, target=target)
    print(sg_miner.summarize(subgroups, target))
    
    if subgroups:
        sg_df = sg_miner.results_to_dataframe(subgroups)
        display(sg_df.head(5))

## 4. Statistical Bias Testing

**Goal:** Systematic hypothesis testing across all demographic group pairs, with proper multiple comparison correction.

In [ ]:
tester = StatisticalBiasTester(
    alpha=0.05,
    correction_method='benjamini-hochberg',
    min_group_size=20,
    effect_size_threshold=0.2  # Lower threshold for demo
)

results = tester.test_pairwise(df)
results_df = tester.results_to_dataframe(results)
print(tester.summarize(results))

In [ ]:
# Show significant findings
sig = results_df[results_df['is_significant']].sort_values('effect_size', ascending=False)
if len(sig) > 0:
    display(sig.head(15))
else:
    print("No significant findings at current thresholds.")
    print("Top 5 by effect size (even if not significant):")
    display(results_df.nlargest(5, 'effect_size'))

## 5. Fairness Metrics

In [ ]:
fm = FairnessMetrics()
summary = fm.summary_table(df)
display(summary)

# Counterfactual gap for gender
print("\nCounterfactual Fairness Gaps (Gender):")
cf_gap = fm.counterfactual_gap(df, 'gender')
display(cf_gap)

## 6. Visualizations

In [ ]:
# Heatmap: toxicity across gender × race
pivot = df.pivot_table(values='toxicity_score', index='gender', columns='race_ethnicity', aggfunc='mean')
plt.figure(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', center=df['toxicity_score'].mean())
plt.title('Mean Toxicity Score: Gender × Race')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of sentiment by race within criminal justice category
cj = df[df['category'] == 'criminal_justice']
plt.figure(figsize=(10, 5))
sns.violinplot(data=cj, x='race_ethnicity', y='sentiment_positive', inner='box')
plt.title('Sentiment Distribution in Criminal Justice Scenarios by Race')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Refusal rate by demographics
refusal_rates = df.groupby(['gender', 'race_ethnicity'])['is_refusal'].mean().unstack()
plt.figure(figsize=(10, 4))
sns.heatmap(refusal_rates, annot=True, fmt='.2%', cmap='YlOrRd')
plt.title('Refusal Rate: Gender × Race')
plt.tight_layout()
plt.show()

## 7. Next Steps

To run this on a **real LLM**:

```python
from src.collectors.prompt_generator import PromptGenerator
from src.collectors.llm_querier import LLMQuerier
from src.metrics.response_analyzer import ResponseAnalyzer

# Generate prompts
gen = PromptGenerator('data/prompts/templates.yaml')
prompts_df = gen.generate_sample(n=500)

# Query model
querier = LLMQuerier(model_name='meta-llama/Llama-3.1-8B-Instruct')
responses_df = querier.collect(prompts_df)

# Analyze with HF pipelines
analyzer = ResponseAnalyzer()
enriched_df = analyzer.analyze(responses_df)

# Then run the same mining methods on enriched_df
```